# Pixabay — Descrição de Imagens via IA

Preenche `Tags_PT`, `Tags_EN`, `Tags_Oracao_PT`, `Tags_Oracao_EN`, `Tags_Biblia_PT`, `Tags_Biblia_EN`, `Descricao_Cena_PT` e `Descricao_Cena_EN` direto na planilha **pixabay-image-stock** (aba `image-stock`), usando Groq e Mistral (visão) com rodízio entre os dois provedores. As tags _livres_ ficam por conta de outra etapa (chat), fora deste notebook.

Roda direto no Google Sheets — não precisa baixar nada manualmente, só rodar as células em ordem.

**Layout novo**: os 8 campos que este notebook preenche ficam em colunas contíguas — grava tudo numa chamada só.

**URL fresca**: a URL de imagem guardada na planilha (coluna `Imagem`) só vale 24h — esse notebook busca uma URL nova direto na API da Pixabay (usando o ID, que é permanente) antes de cada download.

In [1]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  1. DEPENDÊNCIAS E AUTENTICAÇÃO                                  ║
# ╚══════════════════════════════════════════════════════════════════╝
!pip install -q -U groq gspread requests "mistralai>=1.2.0"

import json
import time
from pathlib import Path

import gspread
import requests
from google.auth import default
from google.colab import auth, userdata
from groq import Groq
# Caminho de import oficial e documentado do SDK unificado da Mistral (não é
# "mistralai import Mistral" — veja o próprio exemplo de vision da Mistral em
# https://docs.mistral.ai/capabilities/vision):
from mistralai.client import Mistral

# Autenticação Google Drive / Sheets
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# Chaves via Secrets do Colab (ícone de chave 🔑 na barra lateral esquerda)
GROQ_API_KEY = userdata.get("GROQ_KEY")
MISTRAL_API_KEY = userdata.get("MISTRAL_KEY")
PIXABAY_API_KEY = userdata.get("PIXABAY_KEY")  # gratuita, cadastro em pixabay.com/api/docs

groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None
mistral_client = Mistral(api_key=MISTRAL_API_KEY) if MISTRAL_API_KEY else None

print("=" * 60)
print("✅ SETUP CONCLUÍDO")
print("=" * 60)
print(f"   Groq:    {'disponível' if groq_client else '❌ GROQ_KEY não encontrada nos Secrets'}")
print(f"   Mistral: {'disponível' if mistral_client else '❌ MISTRAL_KEY não encontrada nos Secrets'}")
print(f"   Pixabay: {'disponível' if PIXABAY_API_KEY else '❌ PIXABAY_KEY não encontrada nos Secrets (necessária pra buscar URL fresca de imagem)'}")
print("=" * 60)

✅ SETUP CONCLUÍDO
   Groq:    disponível
   Mistral: disponível
   Pixabay: disponível


In [2]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  2. CONFIGURAÇÃO                                                 ║
# ╚══════════════════════════════════════════════════════════════════╝
ID_PLANILHA = "1P2LydKeeoU5MsAPNl1qhno5qsD1q5BbMOeTbblOVU1E"  # ID fixo da planilha (parte
                            # da URL entre /d/ e /edit) — abre por ID em vez de nome, porque a
                            # planilha está numa conta Google diferente da que autentica esse
                            # notebook (alanabdmorais@gmail.com vs oracaobomdiajesus@gmail.com).
NOME_ABA            = "image-stock"  # aba dentro da planilha

MODELO_GROQ    = "qwen/qwen3.6-27b"          # confirmado em console.groq.com/docs/vision (ago/2026)
MODELO_MISTRAL = "mistral-small-latest"      # alias oficial (docs.mistral.ai/capabilities/vision) — sempre aponta pro modelo de vision atual

DELAY_SEGUNDOS      = 4    # pausa entre linhas, pra não estourar cota de nenhum dos dois provedores
TIMEOUT_DOWNLOAD    = 15   # segundos, download da imagem
MAX_TOKENS_RESPOSTA = 1200  # tokens de saída — margem confortável pros 8 campos

# Quantas linhas AINDA PENDENTES (sem Descricao_Cena_PT) processar nesta
# execução — não conta as que já estão preenchidas. Comece baixo pra
# testar; aumente (ou deixe None pra processar todas as pendentes) depois
# que confirmar que está tudo certo.
LIMITE_LINHAS = 20

print(f"Planilha (ID): {ID_PLANILHA} / aba: {NOME_ABA}")
print(f"Delay entre linhas: {DELAY_SEGUNDOS}s")
print(f"Limite desta execução: {LIMITE_LINHAS if LIMITE_LINHAS is not None else 'sem limite — todas as pendentes'}")

Planilha (ID): 1P2LydKeeoU5MsAPNl1qhno5qsD1q5BbMOeTbblOVU1E / aba: image-stock
Delay entre linhas: 4s
Limite desta execução: 20


In [3]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  3. ABRIR A PLANILHA E DIAGNÓSTICO                               ║
# ╚══════════════════════════════════════════════════════════════════╝
sheet = gc.open_by_key(ID_PLANILHA).worksheet(NOME_ABA)
dados_tabela = sheet.get_all_records()
cabecalho = sheet.row_values(1)


def col(nome):
    """Índice 1-based da coluna pelo nome do cabeçalho — se a ordem das
    colunas mudar na planilha um dia, isso continua funcionando (não
    depende de posição fixa)."""
    return cabecalho.index(nome) + 1


# Layout novo: os 8 campos abaixo ficam em colunas CONTÍGUAS na planilha
# (Tags_PT é a primeira, Descricao_Cena_EN é a última do bloco) — por isso
# só precisamos do índice de início e de fim pra gravar tudo numa
# chamada só (ver gravar_linha, célula 4).
idx_bloco_ini = col("Tags_PT")
idx_bloco_fim = col("Descricao_Cena_EN")

ja_preenchidas = sum(1 for r in dados_tabela if r.get("Descricao_Cena_PT"))
pendentes = len(dados_tabela) - ja_preenchidas

print("=" * 60)
print("📊 DIAGNÓSTICO")
print("=" * 60)
print(f"   Total de linhas: {len(dados_tabela)}")
print(f"   Já preenchidas:  {ja_preenchidas}")
print(f"   Pendentes:       {pendentes}")
print("=" * 60)

📊 DIAGNÓSTICO
   Total de linhas: 105
   Já preenchidas:  105
   Pendentes:       0


In [4]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  4. FUNÇÕES                                                      ║
# ╚══════════════════════════════════════════════════════════════════╝
PROMPT_SISTEMA = """
Analise a imagem e retorne JSON estrito com os campos abaixo.
Respeite os limites à risca — respostas longas demais quebram o pipeline.

- tags_pt: tradução/ajuste das tags originais para PT-BR. NO MÁXIMO as mesmas
  tags originais, sem adicionar novas. Separadas por vírgula, sem frases.
- tags_en: tags originais limpas, sem termos genéricos. MESMO LIMITE de tags_pt.

- tags_oracao_pt: escolha 2 a 5 itens da LISTA FECHADA abaixo que combinem
  com o SENTIMENTO/tema devocional da cena (não repita a descrição visual —
  pense no que essa cena EVOCA espiritualmente). Copie exatamente como
  está escrito na lista. Se nada combinar bem, deixe vazio — não force.
  LISTA FECHADA: paz, súplica, entrega, confiança, esperança, fé, consolo, descanso, refúgio, alegria, humildade, contemplação, silêncio, expectativa, vigília, adoração, louvor, gratidão, celebração, exaltação, reverência, arrependimento, perdão, restauração, renovação, novo começo, cura interior, provisão, proteção, força, fortalecimento, sabedoria, direção, orientação, sustento, presença de deus, comunhão, intimidade, amor, graça, misericórdia, santidade, obediência, chamado, serviço, unidade, intercessão, súplica por outros, unidade familiar, comunhão fraterna, batalha espiritual, vigília em oração, jornada, travessia, deserto interior, tempestade da vida, provação, libertação, vitória espiritual
- tags_oracao_en: tradução direta de cada item escolhido em tags_oracao_pt.

- tags_biblia_pt: escolha 1 a 4 itens da LISTA FECHADA abaixo que essa
  cena poderia ilustrar numa narração de uma história bíblica ESPECÍFICA
  (isso é sobre NARRATIVA, diferente de tags_oracao que é sobre
  sentimento). Copie exatamente como está escrito. Se nada combinar bem
  (a maioria das cenas de banco de imagens NÃO vai ter conexão bíblica
  narrativa clara — tudo bem, é esperado), deixe vazio.
  LISTA FECHADA: criação, jardim do éden, queda, dilúvio, arca de noé, torre de babel, chamado de abraão, aliança, sacrifício de isaque, jacó e esaú, escada de jacó, josé e os irmãos, sonhos proféticos, escravidão no egito, sarça ardente, pragas do egito, travessia do mar vermelho, maná no deserto, dez mandamentos, monte sinai, bezerro de ouro, tabernáculo, arca da aliança, peregrinação no deserto, serpente de bronze, terra prometida, queda de jericó, juízes, sansão, gideão, débora, rute e noemi, davi e golias, davi e saul, reino de davi, sabedoria de salomão, templo de salomão, reino dividido, elias no monte carmelo, carro de fogo, eliseu, exílio babilônico, daniel na cova dos leões, fornalha ardente, jonas e o grande peixe, ester, sofrimento de jó, salmos e louvor, provérbios e sabedoria, reconstrução do templo, profecia messiânica, anunciação, natividade, magos do oriente, estrela de belém, fuga para o egito, apresentação no templo, batismo de jesus, tentação no deserto, chamado dos discípulos, sermão da montanha, bem-aventuranças, milagre de cura, multiplicação dos pães, tempestade acalmada, jesus anda sobre as águas, parábola do semeador, parábola do filho pródigo, parábola do bom samaritano, ovelha perdida, transfiguração, ressurreição de lázaro, entrada triunfal em jerusalém, última ceia, getsêmani, prisão e julgamento, crucificação, ressurreição de jesus, tumba vazia, estrada de emaús, ascensão, pentecostes, conversão de paulo, viagens missionárias, igreja primitiva, perseguição dos cristãos, cartas apostólicas, apocalipse, pastor e ovelhas, boas novas, anjo mensageiro, profeta, rei, sacerdote, juízo, misericórdia divina, aliança renovada, êxodo espiritual, batalha espiritual, jornada de fé, provação, milagre, cura, ressurreição, segunda vinda, reino de deus, cordeiro de deus, luz do mundo
- tags_biblia_en: tradução direta de cada item escolhido em tags_biblia_pt.

- descricao_cena_pt: EXATAMENTE 2 frases curtas e literais (o que se vê, sem
  poesia). Máximo 40 palavras no total.
- descricao_cena_en: tradução da descrição. MESMO LIMITE de 40 palavras.

Responda SÓ o JSON, sem texto antes ou depois. NÃO preencha tags_oracao_livre_*
nem tags_biblia_livre_* — esses campos ficam por conta de outra etapa,
não são pedidos aqui.
"""


def buscar_url_fresca(vid_id):
    """As URLs de imagem que a API da Pixabay devolve (webformatURL/
    largeImageURL) só valem por 24h -- por isso a URL guardada na planilha
    (coluna "Imagem") já pode estar vencida quando esse notebook roda. O ID
    da imagem, esse sim, é permanente -- usamos ele pra pedir uma URL nova
    direto na API antes de cada download.
    Doc oficial: https://pixabay.com/api/docs/"""
    if not PIXABAY_API_KEY:
        raise RuntimeError("PIXABAY_KEY não configurada nos Secrets do Colab")
    resp = requests.get(
        "https://pixabay.com/api/",
        params={"key": PIXABAY_API_KEY, "id": vid_id},
        timeout=TIMEOUT_DOWNLOAD,
    )
    resp.raise_for_status()
    hits = resp.json().get("hits", [])
    if not hits:
        return None
    # webformatURL (640px) é suficiente pra descrição via IA -- mais leve
    # que largeImageURL, sem precisar da imagem em resolução alta
    return hits[0].get("webformatURL") or hits[0].get("largeImageURL")


def baixar_imagem(vid_id):
    url_fresca = buscar_url_fresca(vid_id)
    if not url_fresca:
        print(f"  ❌ ID {vid_id} não encontrado na Pixabay (removido ou inválido)")
        return None
    destino = Path(f"{vid_id}_foto.jpg")
    try:
        resp = requests.get(url_fresca, timeout=TIMEOUT_DOWNLOAD)
        resp.raise_for_status()
    except Exception as e:
        print(f"  ❌ Falha ao baixar imagem (URL fresca): {e}")
        return None
    destino.write_bytes(resp.content)
    return destino


def imagem_para_b64(caminho):
    import base64
    return base64.b64encode(caminho.read_bytes()).decode("utf-8")


def deletar_temp(*caminhos):
    for c in caminhos:
        if c and c.exists():
            try:
                c.unlink()
            except Exception:
                pass


def _extrair_json(texto):
    """Parse tolerante: tira cerca de código markdown (```json ... ```)
    se o modelo colocar uma, e só então tenta json.loads. Usado tanto pro
    Groq quanto pro Mistral -- ver comentário em chamar_groq sobre por que
    isso substituiu o response_format=json_object do Groq."""
    texto = texto.strip()
    if texto.startswith("```"):
        texto = texto.split("```")[1]
        if texto.startswith("json"):
            texto = texto[4:]
        texto = texto.strip()
    return json.loads(texto)


def chamar_groq(prompt, b64_list):
    # Sem response_format=json_object de propósito: a validação ESTRITA do
    # Groq pro modo JSON está com falha de confiabilidade conhecida --
    # aqui a gente só pede JSON no texto do prompt e faz o parse tolerante
    # do nosso lado (_extrair_json).
    content = [{"type": "text", "text": prompt}]
    for img in b64_list:
        content.append({"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{img}"}})
    res = groq_client.chat.completions.create(
        model=MODELO_GROQ,
        messages=[{"role": "user", "content": content}],
        temperature=0.2,
        max_tokens=MAX_TOKENS_RESPOSTA,
    )
    return _extrair_json(res.choices[0].message.content)


def chamar_mistral(prompt, b64_list):
    content = [{"type": "text", "text": prompt}]
    for img in b64_list:
        content.append({"type": "image_url", "image_url": f"data:image/jpeg;base64,{img}"})
    res = mistral_client.chat.complete(
        model=MODELO_MISTRAL,
        messages=[{"role": "user", "content": content}],
        response_format={"type": "json_object"},
        temperature=0.2,
        max_tokens=MAX_TOKENS_RESPOSTA,
    )
    return _extrair_json(res.choices[0].message.content)


def _e_cota_diaria_esgotada(erro):
    """Cota DIÁRIA (TPD — tokens per day) é diferente de cota por MINUTO
    (TPM): a diária só libera de novo depois de horas, não minutos — não
    vale a pena ficar tentando esse provedor de novo a cada linha pelo
    resto da execução."""
    texto = str(erro).lower()
    return "per day" in texto or "(tpd)" in texto or "tpd)" in texto


def _vale_tentar_outro_provedor(erro):
    """Heurística por texto no erro (+ checagem de tipo pro nosso próprio
    parse) — evita depender da classe exata de exceção de cada SDK.

    Cobre os tipos de falha que vale a pena tentar no OUTRO provedor:
    cota estourada (429/rate limit/quota), json_validate_failed (formato
    antigo do Groq, mantido por segurança), e json.JSONDecodeError
    (formato atual — nosso próprio _extrair_json detectando conteúdo
    vazio/malformado)."""
    if isinstance(erro, json.JSONDecodeError):
        return True
    texto = str(erro).lower()
    e_erro_de_cota = any(m in texto for m in ("429", "rate limit", "rate_limit", "too many requests", "quota"))
    e_geracao_vazia = "json_validate_failed" in texto or "failed to validate json" in texto
    return e_erro_de_cota or e_geracao_vazia


def preencher_com_ia(prompt, b64_list, estado):
    """Tenta o provedor 'da vez'; se falhar por limite de cota, tenta
    IMEDIATAMENTE o outro antes de desistir da linha. Se um provedor bater
    na cota DIÁRIA, marca ele como indisponível pro resto da execução."""
    primeiro = estado["atual"]
    segundo = "mistral" if primeiro == "groq" else "groq"
    ultimo_erro = None

    for provedor in (primeiro, segundo):
        if estado.get("indisponivel_ate_o_fim", {}).get(provedor):
            continue
        cliente = groq_client if provedor == "groq" else mistral_client
        if not cliente:
            continue
        try:
            dados = chamar_groq(prompt, b64_list) if provedor == "groq" else chamar_mistral(prompt, b64_list)
            estado["atual"] = segundo if provedor == primeiro else primeiro
            return dados, provedor
        except Exception as e:
            ultimo_erro = e
            print(f"  ⚠️  Falha no {provedor}: {e}")
            if _e_cota_diaria_esgotada(e):
                estado.setdefault("indisponivel_ate_o_fim", {})[provedor] = True
                print(f"  🚫 {provedor}: cota DIÁRIA esgotada — não tenta mais ele nesta execução.")
            if not _vale_tentar_outro_provedor(e):
                break

    estado["atual"] = segundo
    raise ultimo_erro or RuntimeError("Nenhum provedor de IA disponível — confira as chaves nos Secrets ou aguarde a cota liberar")


def normalizar_valor_celula(valor):
    """A IA às vezes devolve um campo como lista em vez de texto único —
    o Google Sheets rejeita lista dentro de uma célula. Junta em string
    nesse caso; qualquer outro tipo inesperado também vira string."""
    if isinstance(valor, (list, tuple)):
        return ", ".join(str(v) for v in valor)
    if valor is None:
        return ""
    return str(valor)


def gravar_linha(num_linha, tags_pt, tags_en, oracao_pt, oracao_en,
                  biblia_pt, biblia_en, desc_pt, desc_en, tentativas=3):
    """Grava os 8 campos numa chamada só — layout novo, as colunas
    Tags_PT...Descricao_Cena_EN ficam contíguas na planilha."""
    valores = [tags_pt, tags_en, oracao_pt, oracao_en, biblia_pt, biblia_en, desc_pt, desc_en]
    celula_ini = gspread.utils.rowcol_to_a1(num_linha, idx_bloco_ini)
    celula_fim = gspread.utils.rowcol_to_a1(num_linha, idx_bloco_fim)
    intervalo = f"{celula_ini}:{celula_fim}"
    for tentativa in range(1, tentativas + 1):
        try:
            sheet.update(values=[valores], range_name=intervalo)
            return True
        except gspread.exceptions.APIError as e:
            print(f"  ⚠️  Erro ao gravar {intervalo} (tentativa {tentativa}/{tentativas}): {e}")
            time.sleep(5 * tentativa)
    return False

In [5]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  5. LOOP PRINCIPAL                                               ║
# ╚══════════════════════════════════════════════════════════════════╝
from tqdm.auto import tqdm

if not (groq_client or mistral_client):
    raise RuntimeError("Nenhuma API disponível — confira GROQ_KEY / MISTRAL_KEY nos Secrets do Colab")

candidatas = []
for num_linha, row in enumerate(dados_tabela, start=2):
    if row.get("Descricao_Cena_PT"):
        continue
    vid_id = str(row.get("ID", "")).strip()
    if not vid_id or vid_id.lower() == "nan":
        continue
    candidatas.append((num_linha, row))
    if LIMITE_LINHAS is not None and len(candidatas) >= LIMITE_LINHAS:
        break

print(f"Processando {len(candidatas)} linha(s) nesta execução...")

estado_provedor = {"atual": "groq"}
processadas, falhas, sem_midia = 0, 0, 0

barra = tqdm(candidatas, unit="linha")
for num_linha, row in barra:
    vid_id = str(row.get("ID"))
    tags_orig = str(row.get("Tags", ""))

    barra.set_postfix_str(f"ID {vid_id}")

    # busca URL fresca na API (a guardada na planilha já pode ter expirado --
    # URLs da Pixabay só valem 24h) e baixa em seguida
    caminho = baixar_imagem(vid_id)
    if not caminho:
        tqdm.write(f"  ❌ [Linha {num_linha}] ID {vid_id}: falha no download")
        sem_midia += 1
        time.sleep(DELAY_SEGUNDOS)
        continue

    b64_list = [imagem_para_b64(caminho)]
    deletar_temp(caminho)

    prompt = f"ID: {vid_id}\nTags Originais: {tags_orig}\n{PROMPT_SISTEMA}"

    try:
        dados, provedor_usado = preencher_com_ia(prompt, b64_list, estado_provedor)
    except Exception as e:
        tqdm.write(f"  ❌ [Linha {num_linha}] ID {vid_id}: falha total (Groq + Mistral): {e}")
        falhas += 1
        time.sleep(DELAY_SEGUNDOS)
        continue

    tags_pt   = normalizar_valor_celula(dados.get("tags_pt", ""))
    tags_en   = normalizar_valor_celula(dados.get("tags_en", ""))
    oracao_pt = normalizar_valor_celula(dados.get("tags_oracao_pt", ""))
    oracao_en = normalizar_valor_celula(dados.get("tags_oracao_en", ""))
    biblia_pt = normalizar_valor_celula(dados.get("tags_biblia_pt", ""))
    biblia_en = normalizar_valor_celula(dados.get("tags_biblia_en", ""))
    desc_pt   = normalizar_valor_celula(dados.get("descricao_cena_pt", ""))
    desc_en   = normalizar_valor_celula(dados.get("descricao_cena_en", ""))

    if gravar_linha(num_linha, tags_pt, tags_en, oracao_pt, oracao_en, biblia_pt, biblia_en, desc_pt, desc_en):
        processadas += 1
    else:
        falhas += 1
        tqdm.write(f"  ❌ [Linha {num_linha}] ID {vid_id}: falha ao gravar na planilha (será retentada na próxima rodada)")

    time.sleep(DELAY_SEGUNDOS)

print()
print("=" * 60)
print("🎉 PROCESSAMENTO CONCLUÍDO")
print("=" * 60)
print(f"   ✅ Preenchidas: {processadas}")
print(f"   ⚠️  Sem mídia:   {sem_midia}")
print(f"   ❌ Falhas:      {falhas}")
print("=" * 60)

Processando 0 linha(s) nesta execução...


0linha [00:00, ?linha/s]


🎉 PROCESSAMENTO CONCLUÍDO
   ✅ Preenchidas: 0
   ⚠️  Sem mídia:   0
   ❌ Falhas:      0
